# H18 - Image and Audio Input Capability Probe

This notebook only checks whether Gemma E2B/E4B can receive direct image and audio inputs through the Hugging Face processor/model stack.

Each case asks for a neutral description or transcription so the output can reveal whether the media reached the model.

Datasets used by default:

- Images: `beans`, a compact leaf-image dataset whose rows include a PIL `image` column.
- Audio: `hf-internal-testing/librispeech_asr_dummy`, a tiny LibriSpeech-style validation split with an `audio` column and reference `text`.

Outputs are written to:

```text
/content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_results.csv
/content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_metrics.csv
/content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_decision.md
```

## Install

Use a Colab GPU runtime for real model inference. CPU is only practical with `RUN_MODEL_INFERENCE = False` for a contract smoke test.

In [3]:
# Keep the model stack aligned with the other Gemma notebooks, with datasets extras for media fixtures.
%pip install -q --upgrade --extra-index-url https://download.pytorch.org/whl/cu128 \
  git+https://github.com/huggingface/transformers.git \
  torch torchvision torchaudio \
  accelerate \
  bitsandbytes \
  tokenizers \
  huggingface_hub \
  safetensors \
  datasets[audio,vision] \
  pandas==2.2.2 \
  numpy==1.26.4 \
  pillow \
  soundfile \
  librosa

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg libavcodec-dev libavformat-dev libavutil-dev libswresample-dev
!pip install -U torchcodec==0.11.1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## Persistent Paths and Configuration

In [13]:
from pathlib import Path
import random
import numpy as np
import torch
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

SEED = 0
random.seed(SEED)
np.random.seed(SEED)

NOTEBOOKS_ROOT = Path("/content/drive/MyDrive/GemScan/notebooks")
RESULTS_DIR = NOTEBOOKS_ROOT / "_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV_PATH = RESULTS_DIR / "h18_audio_video_capability_results.csv"
METRICS_CSV_PATH = RESULTS_DIR / "h18_audio_video_capability_metrics.csv"
DECISION_PATH = RESULTS_DIR / "h18_audio_video_capability_decision.md"

MODEL_TIERS = {
    "E2B": "google/gemma-4-E2B-it",
    # Uncomment when you want to compare the larger tier.
    # "E4B": "google/gemma-4-E4B-it",
}

RUN_MODEL_INFERENCE = True
USE_4BIT = True
FOUR_BIT_COMPUTE_DTYPE = torch.float32
RESET_RESULTS = True
MAX_NEW_TOKENS = 96
NUM_IMAGE_CASES = 5
NUM_AUDIO_CASES = 5

IMAGE_DATASET_ID = "beans"
IMAGE_DATASET_FALLBACK_ID = "JohnHapsara/beans"
IMAGE_DATASET_SPLIT = "validation"

AUDIO_DATASET_ID = "hf-internal-testing/librispeech_asr_dummy"
AUDIO_DATASET_SPLIT = "validation"
AUDIO_SAMPLING_RATE = 16000

CAPABILITY_SCHEMA_VERSION = "h18_media_input_capability_v2"

print("RESULTS_CSV_PATH", RESULTS_CSV_PATH)
print("RUN_MODEL_INFERENCE", RUN_MODEL_INFERENCE)
print("MODEL_TIERS", MODEL_TIERS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RESULTS_CSV_PATH /content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_results.csv
RUN_MODEL_INFERENCE True
MODEL_TIERS {'E2B': 'google/gemma-4-E2B-it'}


## Optional Hugging Face Login

Run this if model or dataset download fails with authentication, license, or rate-limit errors.

In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load Gemma Tiers

In [14]:
models = {}
processors = {}

if RUN_MODEL_INFERENCE:
    import torch
    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

    for model_tier, model_id in MODEL_TIERS.items():
        print(f"Loading {model_tier}: {model_id}")
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

        kwargs = {
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
            "device_map": "auto",
        }

        if torch.cuda.is_available():
            if USE_4BIT:
                kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=FOUR_BIT_COMPUTE_DTYPE,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=False,
                )
                kwargs["dtype"] = FOUR_BIT_COMPUTE_DTYPE
            else:
                kwargs["dtype"] = torch.bfloat16
        else:
            kwargs["dtype"] = torch.float32

        model = AutoModelForMultimodalLM.from_pretrained(model_id, **kwargs)
        model.eval()

        processors[model_tier] = processor
        models[model_tier] = model
        print(f"Loaded {model_tier}")
else:
    print("Skipping model load (RUN_MODEL_INFERENCE=False).")

Loading E2B: google/gemma-4-E2B-it


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Loaded E2B


## Pull 5 Image Cases and 5 Audio Cases

These are neutral capability fixtures. Labels/transcripts are kept only as metadata; prompts only ask the model to describe or transcribe the media.

In [15]:
from huggingface_hub import hf_hub_download, list_repo_files
from io import BytesIO
from datasets import load_dataset
import pandas as pd
import soundfile as sf


def load_image_dataset():
    try:
        return load_dataset(IMAGE_DATASET_ID, split=IMAGE_DATASET_SPLIT), IMAGE_DATASET_ID
    except Exception as exc:
        print(f"Could not load {IMAGE_DATASET_ID!r}: {exc}")
        return load_dataset(IMAGE_DATASET_FALLBACK_ID, split=IMAGE_DATASET_SPLIT), IMAGE_DATASET_FALLBACK_ID


def load_audio_parquet(dataset_id: str, split: str) -> pd.DataFrame:
    """Load audio rows as plain Parquet so datasets.Audio never invokes TorchCodec."""
    repo_files = list_repo_files(dataset_id, repo_type="dataset")
    parquet_files = [name for name in repo_files if name.endswith(".parquet")]
    split_matches = [name for name in parquet_files if split.lower() in name.lower()]
    candidates = split_matches or parquet_files
    if not candidates:
        raise FileNotFoundError(f"No parquet files found in dataset repo {dataset_id}")

    parquet_name = sorted(candidates)[0]
    parquet_path = hf_hub_download(repo_id=dataset_id, repo_type="dataset", filename=parquet_name)
    print(f"Loaded audio parquet file: {parquet_name}")
    return pd.read_parquet(parquet_path)


def decode_audio_without_torchcodec(audio_value):
    """Decode Parquet audio bytes/path with soundfile instead of datasets.Audio/TorchCodec."""
    if isinstance(audio_value, dict):
        audio_bytes = audio_value.get("bytes")
        audio_path = audio_value.get("path")
    else:
        audio_bytes = getattr(audio_value, "bytes", None)
        audio_path = getattr(audio_value, "path", None)

    if audio_bytes is not None:
        array, sampling_rate = sf.read(BytesIO(audio_bytes), dtype="float32", always_2d=False)
        source_path = audio_path or ""
    elif audio_path:
        array, sampling_rate = sf.read(audio_path, dtype="float32", always_2d=False)
        source_path = audio_path
    else:
        raise ValueError(f"Unsupported audio value from parquet: {type(audio_value)!r}")

    if array.ndim > 1:
        array = array.mean(axis=1)
    return {"array": array, "sampling_rate": sampling_rate, "path": source_path}


image_ds, resolved_image_dataset_id = load_image_dataset()
audio_df = load_audio_parquet(AUDIO_DATASET_ID, AUDIO_DATASET_SPLIT)

image_cases = []
for index in range(min(NUM_IMAGE_CASES, len(image_ds))):
    row = image_ds[index]
    label_feature = image_ds.features.get("labels")
    label_value = row.get("labels", "")
    label_name = label_feature.int2str(label_value) if hasattr(label_feature, "int2str") else str(label_value)
    image_cases.append(
        {
            "case_id": f"image_{index:03d}",
            "modality": "image",
            "dataset_id": resolved_image_dataset_id,
            "dataset_split": IMAGE_DATASET_SPLIT,
            "dataset_index": index,
            "media": row["image"].convert("RGB"),
            "metadata": {"label": label_name},
            "prompt": "Describe this image in one short sentence.",
        }
    )

audio_cases = []
for index in range(min(NUM_AUDIO_CASES, len(audio_df))):
    row = audio_df.iloc[index]
    audio = decode_audio_without_torchcodec(row["audio"])
    audio_cases.append(
        {
            "case_id": f"audio_{index:03d}",
            "modality": "audio",
            "dataset_id": AUDIO_DATASET_ID,
            "dataset_split": AUDIO_DATASET_SPLIT,
            "dataset_index": index,
            "media": audio,
            "metadata": {"reference_text": row.get("text", ""), "id": row.get("id", "")},
            "prompt": "Transcribe or summarize this audio in one short sentence. If no audio is available, say so.",
        }
    )

media_cases = image_cases + audio_cases
print(f"Loaded {len(image_cases)} image cases from {resolved_image_dataset_id}:{IMAGE_DATASET_SPLIT}")
print(f"Loaded {len(audio_cases)} audio cases from {AUDIO_DATASET_ID}:{AUDIO_DATASET_SPLIT}")
display(pd.DataFrame([
    {
        "case_id": case["case_id"],
        "modality": case["modality"],
        "dataset_id": case["dataset_id"],
        "dataset_index": case["dataset_index"],
        "metadata": case["metadata"],
    }
    for case in media_cases
]))

Loaded audio parquet file: clean/validation-00000-of-00001.parquet
Loaded 5 image cases from beans:validation
Loaded 5 audio cases from hf-internal-testing/librispeech_asr_dummy:validation


,case_id,modality,dataset_id,dataset_index,metadata
0,image_000,image,beans,0,{'label': 'angular_leaf_spot'}
1,image_001,image,beans,1,{'label': 'angular_leaf_spot'}
2,image_002,image,beans,2,{'label': 'angular_leaf_spot'}
3,image_003,image,beans,3,{'label': 'angular_leaf_spot'}
4,image_004,image,beans,4,{'label': 'angular_leaf_spot'}
5,audio_000,audio,hf-internal-testing/librispeech_asr_dummy,0,{'reference_text': 'MISTER QUILTER IS THE APOS...
6,audio_001,audio,hf-internal-testing/librispeech_asr_dummy,1,{'reference_text': 'NOR IS MISTER QUILTER'S MA...
7,audio_002,audio,hf-internal-testing/librispeech_asr_dummy,2,{'reference_text': 'HE TELLS US THAT AT THIS F...
8,audio_003,audio,hf-internal-testing/librispeech_asr_dummy,3,{'reference_text': 'HE HAS GRAVE DOUBTS WHETHE...
9,audio_004,audio,hf-internal-testing/librispeech_asr_dummy,4,{'reference_text': 'LINNELL'S PICTURES ARE A S...


## Direct Media Capability Helpers

In [16]:
import json
import traceback
import tempfile
import soundfile as sf
import torch
import numpy as np
import os
from typing import Dict

def processor_capability_summary(processor) -> str:
    attrs = [
        name for name in [
            "image_processor",
            "video_processor",
            "audio_processor",
            "feature_extractor",
            "tokenizer",
        ]
        if hasattr(processor, name)
    ]
    return ",".join(attrs) if attrs else processor.__class__.__name__


def move_inputs_to_model(inputs, model):
    try:
        return inputs.to(model.device)
    except AttributeError:
        return {
            key: value.to(model.device) if hasattr(value, "to") else value
            for key, value in inputs.items()
        }


def tensor_key_summary(processed) -> str:
    if not isinstance(processed, dict):
        return ""

    parts = []
    for key, value in processed.items():
        shape = tuple(value.shape) if hasattr(value, "shape") else type(value).__name__
        parts.append(f"{key}:{shape}")

    return "; ".join(parts)


def apply_chat_template(processor, messages):
    return processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )


def audio_case_to_wav_path(case: Dict) -> str:
    audio = case["media"]
    candidate_path = audio.get("path", "")

    # Trust only real local files or actual URLs.
    if isinstance(candidate_path, str) and candidate_path:
        if candidate_path.startswith(("http://", "https://")):
            return candidate_path
        if os.path.exists(candidate_path):
            return candidate_path

    # Otherwise write decoded audio array to a real local WAV file.
    array = np.asarray(audio["array"], dtype=np.float32)

    # Ensure mono.
    if array.ndim > 1:
        array = array.mean(axis=1)

    # Light normalization guard.
    peak = np.max(np.abs(array)) if array.size else 0
    if peak > 1.0:
        array = array / peak

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as wav:
        wav_path = wav.name

    sf.write(
        wav_path,
        array,
        int(audio["sampling_rate"]),
        subtype="PCM_16",
    )

    if not os.path.exists(wav_path):
        raise FileNotFoundError(f"Failed to create temp WAV: {wav_path}")

    return wav_path




def build_messages(case: Dict):
    if case["modality"] == "image":
        return [{
            "role": "user",
            "content": [
                {"type": "image", "image": case["media"]},
                {"type": "text", "text": case["prompt"]},
            ],
        }]

    if case["modality"] == "audio":
        audio_path = audio_case_to_wav_path(case)

        return [{
            "role": "user",
            "content": [
                {"type": "audio", "audio": audio_path},
                {"type": "text", "text": (
                    "Transcribe the following speech segment in English into English text. "
                    "Only output the transcription, with no extra explanation."
                )},
            ],
        }]

    raise ValueError(f"Unsupported modality: {case['modality']}")


def process_case(processor, case: Dict):
    return apply_chat_template(processor, build_messages(case))


def run_media_case(case: Dict, model_tier: str) -> Dict:
    base = {
        "benchmark_schema_version": CAPABILITY_SCHEMA_VERSION,
        "model_tier": model_tier,
        "model_id": MODEL_TIERS[model_tier],
        "case_id": case["case_id"],
        "modality": case["modality"],
        "dataset_id": case["dataset_id"],
        "dataset_split": case["dataset_split"],
        "dataset_index": case["dataset_index"],
        "prompt": case["prompt"],
        "metadata": json.dumps(case["metadata"]),
    }

    if not RUN_MODEL_INFERENCE:
        return {
            **base,
            "status": "skipped_no_model",
            "error": "RUN_MODEL_INFERENCE is False",
            "output_tokens": 0,
            "raw_output": "",
            "processor_summary": "",
            "input_summary": "",
        }

    processor = processors[model_tier]
    model = models[model_tier]

    try:
        processed = process_case(processor, case)
        input_summary = tensor_key_summary(processed)

        device_inputs = move_inputs_to_model(processed, model)

        input_len = (
            device_inputs["input_ids"].shape[-1]
            if "input_ids" in device_inputs
            else 0
        )

        pad_token_id = (
            processor.tokenizer.eos_token_id
            if hasattr(processor, "tokenizer") and processor.tokenizer.eos_token_id is not None
            else None
        )

        generate_kwargs = {
            **device_inputs,
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": False,
        }

        if pad_token_id is not None:
            generate_kwargs["pad_token_id"] = pad_token_id

        with torch.inference_mode():
            output = model.generate(**generate_kwargs)

        raw = processor.decode(
            output[0][input_len:],
            skip_special_tokens=True,
        ).strip()

        output_tokens = (
            len(processor.tokenizer.encode(raw, add_special_tokens=False))
            if hasattr(processor, "tokenizer")
            else 0
        )

        return {
            **base,
            "status": "supported",
            "error": "",
            "output_tokens": output_tokens,
            "raw_output": raw,
            "processor_summary": processor_capability_summary(processor),
            "input_summary": input_summary,
        }

    except Exception:
        return {
            **base,
            "status": "probe_error",
            "error": traceback.format_exc(limit=5),
            "output_tokens": 0,
            "raw_output": "",
            "processor_summary": processor_capability_summary(processor),
            "input_summary": "",
        }

## Run Capability Probe

In [17]:
if RESET_RESULTS and RESULTS_CSV_PATH.exists():
    RESULTS_CSV_PATH.unlink()
    print("deleted existing results", RESULTS_CSV_PATH)

rows = []
for model_tier in MODEL_TIERS:
    for case in media_cases:
        result = run_media_case(case, model_tier)
        rows.append(result)
        print(model_tier, case["case_id"], case["modality"], result["status"])

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_CSV_PATH, index=False)
print("saved final results", RESULTS_CSV_PATH, results_df.shape)
display(results_df[["model_tier", "case_id", "modality", "dataset_id", "dataset_index", "status", "input_summary", "raw_output"]])

deleted existing results /content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_results.csv


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


E2B image_000 image supported
E2B image_001 image supported
E2B image_002 image supported
E2B image_003 image supported
E2B image_004 image supported
E2B audio_000 audio probe_error
E2B audio_001 audio probe_error
E2B audio_002 audio probe_error
E2B audio_003 audio probe_error
E2B audio_004 audio probe_error
saved final results /content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_results.csv (10, 16)


,model_tier,case_id,modality,dataset_id,dataset_index,status,input_summary,raw_output
0,E2B,image_000,image,beans,0,supported,,This image is a repeating pattern of a stylize...
1,E2B,image_001,image,beans,1,supported,,"This is a grayscale image with a repeating, fa..."
2,E2B,image_002,image,beans,2,supported,,"This image is a collage of multiple, overlappi..."
3,E2B,image_003,image,beans,3,supported,,The image is a mostly gray background with a f...
4,E2B,image_004,image,beans,4,supported,,"The image is a mostly gray, abstract visual wi..."
5,E2B,audio_000,audio,hf-internal-testing/librispeech_asr_dummy,0,probe_error,,
6,E2B,audio_001,audio,hf-internal-testing/librispeech_asr_dummy,1,probe_error,,
7,E2B,audio_002,audio,hf-internal-testing/librispeech_asr_dummy,2,probe_error,,
8,E2B,audio_003,audio,hf-internal-testing/librispeech_asr_dummy,3,probe_error,,
9,E2B,audio_004,audio,hf-internal-testing/librispeech_asr_dummy,4,probe_error,,


### Diagnostic Test

In [7]:
for case in audio_cases[:5]:
    audio = case["media"]
    print(case["case_id"])
    print("path:", repr(audio.get("path")))
    print("path exists:", os.path.exists(audio.get("path", "")))
    print("array shape:", np.asarray(audio["array"]).shape)
    print("sampling rate:", audio["sampling_rate"])
    print()

audio_000
path: '1272-128104-0000.flac'
path exists: False
array shape: (93680,)
sampling rate: 16000

audio_001
path: '1272-128104-0001.flac'
path exists: False
array shape: (77040,)
sampling rate: 16000

audio_002
path: '1272-128104-0002.flac'
path exists: False
array shape: (199760,)
sampling rate: 16000

audio_003
path: '1272-128104-0003.flac'
path exists: False
array shape: (158400,)
sampling rate: 16000

audio_004
path: '1272-128104-0004.flac'
path exists: False
array shape: (470400,)
sampling rate: 16000



In [8]:
# Sanity check: did our helper actually create real WAV files?
for case in audio_cases[:5]:
    wav_path = audio_case_to_wav_path(case)
    print(case["case_id"])
    print("wav_path:", repr(wav_path))
    print("exists:", os.path.exists(wav_path))
    print("size bytes:", os.path.getsize(wav_path) if os.path.exists(wav_path) else None)
    print()

audio_000
wav_path: '/tmp/tmphftwgk7o.wav'
exists: True
size bytes: 187404

audio_001
wav_path: '/tmp/tmpe1uxy2wt.wav'
exists: True
size bytes: 154124

audio_002
wav_path: '/tmp/tmpymdv4wp3.wav'
exists: True
size bytes: 399564

audio_003
wav_path: '/tmp/tmpzwe8gqkz.wav'
exists: True
size bytes: 316844

audio_004
wav_path: '/tmp/tmpvqh_meot.wav'
exists: True
size bytes: 940844



In [14]:
wav_path = audio_case_to_wav_path(audio_cases[0])

from torchcodec.decoders import AudioDecoder

decoder = AudioDecoder(wav_path)
samples = decoder.get_all_samples()

print("sample rate:", samples.sample_rate)
print("data shape:", samples.data.shape)

sample rate: 16000
data shape: torch.Size([1, 93680])


In [18]:
print(
    results_df.loc[
        results_df["status"].eq("probe_error"),
        ["case_id", "modality", "error"]
    ].to_string(max_colwidth=4000)
)

     case_id modality                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

## Metrics and Decision Output

In [ ]:
results_df = pd.read_csv(RESULTS_CSV_PATH).fillna("")
metrics_rows = []

for (model_tier, modality), frame in results_df.groupby(["model_tier", "modality"]):
    counts = frame["status"].value_counts().to_dict()
    metrics_rows.append(
        {
            "model_tier": model_tier,
            "model_id": frame["model_id"].iloc[0],
            "modality": modality,
            "rows": len(frame),
            "supported_rows": counts.get("supported", 0),
            "probe_error_rows": counts.get("probe_error", 0),
            "skipped_rows": sum(value for key, value in counts.items() if str(key).startswith("skipped")),
            "status_counts": json.dumps(counts, sort_keys=True),
        }
    )

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(METRICS_CSV_PATH, index=False)
print("saved metrics", METRICS_CSV_PATH)
display(metrics_df)


def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "_No rows._"
    markdown_frame = frame.fillna("").astype(str)
    columns = list(markdown_frame.columns)
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for _, row in markdown_frame.iterrows():
        values = [str(row[column]).replace("|", "\\|") for column in columns]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

image_supported = bool(((results_df["modality"] == "image") & (results_df["status"] == "supported")).any())
audio_supported = bool(((results_df["modality"] == "audio") & (results_df["status"] == "supported")).any())

decision_lines = [
    "# H18 Image and Audio Input Capability Decision",
    "",
    f"Results: `{RESULTS_CSV_PATH}`",
    f"Metrics: `{METRICS_CSV_PATH}`",
    f"Image dataset: `{resolved_image_dataset_id}` split `{IMAGE_DATASET_SPLIT}` cases `{NUM_IMAGE_CASES}`",
    f"Audio dataset: `{AUDIO_DATASET_ID}` split `{AUDIO_DATASET_SPLIT}` cases `{NUM_AUDIO_CASES}`",
    f"Model tiers: `{', '.join(f'{tier}={model_id}' for tier, model_id in MODEL_TIERS.items())}`",
    f"Direct image input supported in this run: `{image_supported}`",
    f"Direct audio input supported in this run: `{audio_supported}`",
    "",
    "## Metrics",
    "",
    dataframe_to_markdown(metrics_df),
    "",
    "## Handoff",
    "",
    "- Use rows with `status == supported` and non-empty media tensor keys in `input_summary` as evidence that direct media reached the model stack.",
    "- Treat `probe_error` rows as ingestion failures for the exact processor/model/runtime combination under test.",
    "- This notebook performs neutral media description/transcription only.",
]

DECISION_PATH.write_text("\n".join(decision_lines))
print(DECISION_PATH.read_text())
print("saved decision", DECISION_PATH)

saved metrics /content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_metrics.csv


,model_tier,model_id,modality,rows,supported_rows,probe_error_rows,skipped_rows,status_counts
0,E2B,google/gemma-4-E2B-it,audio,5,0,5,0,"{""probe_error"": 5}"
1,E2B,google/gemma-4-E2B-it,image,5,5,0,0,"{""supported"": 5}"


# H18 Image and Audio Input Capability Decision

Results: `/content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_results.csv`
Metrics: `/content/drive/MyDrive/GemScan/notebooks/_results/h18_audio_video_capability_metrics.csv`
Image dataset: `beans` split `validation` cases `5`
Audio dataset: `hf-internal-testing/librispeech_asr_dummy` split `validation` cases `5`
Model tiers: `E2B=google/gemma-4-E2B-it`
Direct image input supported in this run: `True`
Direct audio input supported in this run: `False`

## Metrics

| model_tier | model_id | modality | rows | supported_rows | probe_error_rows | skipped_rows | status_counts |
| --- | --- | --- | --- | --- | --- | --- | --- |
| E2B | google/gemma-4-E2B-it | audio | 5 | 0 | 5 | 0 | {"probe_error": 5} |
| E2B | google/gemma-4-E2B-it | image | 5 | 5 | 0 | 0 | {"supported": 5} |

## Handoff

- Use rows with `status == supported` and non-empty media tensor keys in `input_summary` as evidence that direct media reached the m

In [ ]:
pd.DataFrame([
    {
        "case_id": case["case_id"],
        "seconds": len(case["media"]["array"]) / case["media"]["sampling_rate"],
        "sampling_rate": case["media"]["sampling_rate"],
    }
    for case in audio_cases
])


,case_id,seconds,sampling_rate
0,audio_000,5.855,16000
1,audio_001,4.815,16000
2,audio_002,12.485,16000
3,audio_003,9.900,16000
4,audio_004,29.400,16000
